# Preprint Critic — Kaggle inference (baseline vs LoRA)

Generates peer reviews from full paper text with the **base Qwen3-4B-Instruct-2507** and with the
**Preprint Critic V3 LoRA adapter**, using the same prompt and decoding settings as training.

## Before you run

1. **Settings → Accelerator:** `GPU T4 x2` (one T4 is enough; the notebook uses GPU 0 only).
2. **Settings → Internet:** on (the base model and, if needed, the adapter repository are downloaded).
3. **Add Data → Upload** `test_dataset_untokenized.jsonl` (about 12 MB) as a Kaggle dataset for
   the full 172-paper test split. If you skip this, the notebook downloads a committed 25-paper
   sample from GitHub instead, so it still runs end to end.
4. Optional: add an `HF_TOKEN` Kaggle secret if the base-model download is rate-limited.

If the base model fails to load with an unknown architecture, run a new cell with
`%pip install -q "transformers==5.19.0" "peft==0.21.2" "accelerate==1.15.0"`, restart the kernel,
and re-run. The version cell prints what is actually installed.

Then **Run All**. Outputs are written to `/kaggle/working/preprint_critic_outputs` and are captured
when you **Save Version**.

## What is fixed to training

- Prompt: the five-paragraph instruction brief, then `Paper:\n` and the paper text.
- Paper budget: 16,050 tokens, using the same `clean_text` and `trim_to_tokens` steps.
- Chat template: the adapter tokenizer, `add_generation_prompt=True` (identical to the base template).
- Sampling: temperature 0.7, top-p 0.8, top-k 20, repetition penalty 1.1.
- Baseline and adapter use the same prompt, seed, and `max_new_tokens`.

The notebook is inference-only. It never trains and never modifies the frozen training notebook.


In [ ]:
from pathlib import Path
import os

# ----------------------------- run configuration -----------------------------
SMOKE_TEST = False          # True: 1 paper, 64 new tokens, to check the pipeline
N_PAPERS = 25               # unique held-out papers; the committed sample has 25
PAPER_IDS = []              # optional explicit paper_id list; empty = random N_PAPERS
MAX_PAPER_TOKENS = 16_050   # same paper budget as training
MAX_NEW_TOKENS = 4_096      # 4,096 covers the base model's ~2,300-token reviews
DO_SAMPLE = True
TEMPERATURE = 0.7
TOP_P = 0.8
TOP_K = 20
REPETITION_PENALTY = 1.1
RUN_BASELINE = True
RUN_ADAPTER = True
RESUME = True               # skip (paper_id, arm) pairs already in reviews.jsonl
SEED = 42

BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
REPO_URL = "https://github.com/HAT-ZAID/Preprint-Critic.git"
SAMPLE_URL = "https://raw.githubusercontent.com/HAT-ZAID/Preprint-Critic/main/comparison/kaggle_sample_papers.jsonl"

WORK_DIR = Path("/kaggle/working")
OUTPUT_DIR = WORK_DIR / "preprint_critic_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REVIEWS_PATH = OUTPUT_DIR / "reviews.jsonl"
CONFIG_PATH = OUTPUT_DIR / "run_config.json"

# Optional overrides if auto-discovery picks the wrong file or folder.
DATA_JSONL = os.environ.get("DATA_JSONL", "")
ADAPTER_DIR_OVERRIDE = os.environ.get("ADAPTER_DIR", "")

if SMOKE_TEST:
    N_PAPERS = 1
    MAX_NEW_TOKENS = 64
    print("SMOKE_TEST on: 1 paper, 64 new tokens")

print(f"papers={N_PAPERS}  paper_tokens={MAX_PAPER_TOKENS}  new_tokens={MAX_NEW_TOKENS}")
print(f"sampling={DO_SAMPLE}  T={TEMPERATURE}  top_p={TOP_P}  top_k={TOP_K}  rep={REPETITION_PENALTY}")
print(f"outputs -> {OUTPUT_DIR}")


In [ ]:
import platform
import subprocess
import sys

import torch
import transformers

for package, module in (("peft==0.21.2", "peft"), ("accelerate==1.15.0", "accelerate")):
    try:
        __import__(module)
    except ImportError:
        print("installing", package)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import peft
import accelerate

print("python", sys.version.split()[0], "|", platform.platform())
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"gpu {i}: {p.name}  {p.total_memory / 2**30:.1f} GiB  sm{p.major}{p.minor}")
    print("bf16 reported:", torch.cuda.is_bf16_supported(), "(T4 emulates bf16; this notebook uses fp16)")
assert torch.cuda.is_available(), "Turn on Settings -> Accelerator -> GPU T4 x2"

import shutil
free_gib = shutil.disk_usage("/kaggle/working").free / 2**30
print(f"free /kaggle/working: {free_gib:.1f} GiB")
assert free_gib > 12, "Need at least 12 GiB free for the fp16 base model and outputs"


In [ ]:
import subprocess

def find_data_jsonl():
    if DATA_JSONL:
        p = Path(DATA_JSONL)
        assert p.is_file(), f"DATA_JSONL does not exist: {p}"
        return p
    for root in (Path("/kaggle/input"), WORK_DIR):
        if not root.exists():
            continue
        hits = sorted(root.rglob("test_dataset_untokenized.jsonl"))
        if hits:
            return hits[0]
        hits = sorted(root.rglob("*untokenized*.jsonl"))
        if hits:
            return hits[0]
        hits = sorted(root.rglob("kaggle_sample_papers.jsonl"))
        if hits:
            return hits[0]
    print("no local test split found; downloading the committed 25-paper sample")
    import urllib.request
    dest = WORK_DIR / "kaggle_sample_papers.jsonl"
    urllib.request.urlretrieve(SAMPLE_URL, dest)
    return dest

def find_adapter_dir():
    if ADAPTER_DIR_OVERRIDE:
        p = Path(ADAPTER_DIR_OVERRIDE)
        assert (p / "adapter_config.json").is_file(), f"no adapter_config.json in {p}"
        return p
    for p in sorted(Path("/kaggle/input").rglob("adapter_config.json")):
        if "rc_project_adapter" in str(p).lower() or "preprint" in str(p).lower():
            return p.parent
    repo = WORK_DIR / "Preprint-Critic"
    target = repo / "rc_project_adapter_V3"
    if not (target / "adapter_config.json").is_file():
        print("adapter not attached; cloning the public repository")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
    assert (target / "adapter_config.json").is_file(), target
    return target

DATA_PATH = find_data_jsonl()
ADAPTER_DIR = find_adapter_dir()
print("data:", DATA_PATH, f"({DATA_PATH.stat().st_size / 2**20:.1f} MiB)")
print("adapter:", ADAPTER_DIR)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

try:
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, token=HF_TOKEN)
    print("tokenizer:", ADAPTER_DIR)
except Exception as exc:
    print("adapter tokenizer failed; using the base tokenizer:", exc)
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = 151643
print("pad", tokenizer.pad_token_id, "eos", tokenizer.eos_token_id)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
    token=HF_TOKEN,
)
model.eval()
model.generation_config.use_cache = True
model.config.use_cache = True

def gpu_mem():
    if not torch.cuda.is_available():
        return 0.0, 0.0
    return torch.cuda.memory_allocated() / 2**30, torch.cuda.memory_reserved() / 2**30

allocated, reserved = gpu_mem()
print(f"loaded on {model.device}; allocated {allocated:.2f} GiB, reserved {reserved:.2f} GiB")


In [ ]:
INSTRUCTIONS = '''You are an independent academic peer reviewer.

Write a substantive, fair, constructive review of the submitted
manuscript. Ground factual claims in the manuscript. Never invent
results, citations, numerical values, or section, table, or figure
references that are not present.

Prioritize the most consequential issues rather than listing generic
objections. For each major concern, identify the relevant claim,
method, or experiment; explain the evidence gap or reasoning behind
the concern and why it matters; and give an actionable question or
suggestion where appropriate.

Assess the contribution and novelty in the context of related work
actually discussed in the manuscript, methodological validity,
whether experiments support the conclusions, baseline adequacy,
limitations, and reproducibility where assessable.

Distinguish demonstrated problems from questions where the evidence
is insufficient. Avoid boilerplate praise, repeated summaries, and
criticisms that could apply to almost any paper. Use natural
peer-review prose. Use headings or bullets when helpful, but do not
force a fixed template or add a verdict unsupported by the evidence.
Be concise without sacrificing substantive, paper-specific analysis.'''

def clean_text(value):
    lines = (value or "").splitlines()
    text = "\n".join(" ".join(line.split()) for line in lines).strip()
    while "\n\n\n" in text:
        text = text.replace("\n\n\n", "\n")
    return text

def trim_to_tokens(text, max_tokens):
    ids = tokenizer(
        text or "",
        add_special_tokens=False,
        truncation=True,
        max_length=max_tokens,
    )["input_ids"]
    return tokenizer.decode(ids, skip_special_tokens=True)

def build_prompt(paper_text):
    paper = trim_to_tokens(clean_text(paper_text), MAX_PAPER_TOKENS)
    return INSTRUCTIONS + "\n\nPaper:\n" + paper

def render(prompt):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

probe = render(build_prompt("A short probe paper."))
print("rendered tail:", repr(tokenizer.decode(probe["input_ids"][0][-30:])))


In [ ]:
import hashlib
import re
import time

def seed_for(paper_id):
    return int(hashlib.sha256(str(paper_id).encode()).hexdigest()[:8], 16) % (2**31)

def _lcs(a, b):
    if not a or not b:
        return 0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b, 1):
            cur.append(prev[j - 1] + 1 if x == y else max(prev[j], cur[j - 1]))
        prev = cur
    return prev[-1]

def rouge_l(hypothesis, reference):
    a = re.findall(r"\w+", (hypothesis or "").lower())
    b = re.findall(r"\w+", (reference or "").lower())
    if not a or not b:
        return 0.0
    lcs = _lcs(a, b)
    p = lcs / len(a)
    r = lcs / len(b)
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)

def repeat5(text):
    toks = (text or "").split()
    if len(toks) < 5:
        return 0.0
    grams = [tuple(toks[i:i + 5]) for i in range(len(toks) - 4)]
    return 1.0 - len(set(grams)) / len(grams)

@torch.inference_mode()
def generate_review(active_model, prompt, seed):
    enc = render(prompt)
    input_ids = enc["input_ids"].to(active_model.device)
    attention_mask = enc["attention_mask"].to(active_model.device)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    kwargs = dict(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    if DO_SAMPLE:
        kwargs.update(
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            repetition_penalty=REPETITION_PENALTY,
        )
    else:
        kwargs["do_sample"] = False
    started = time.time()
    out = active_model.generate(**kwargs)
    seconds = time.time() - started
    new_ids = out[0, input_ids.shape[1]:].cpu()
    text = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    return {
        "review": text,
        "prompt_tokens": int(input_ids.shape[1]),
        "output_tokens": int(new_ids.shape[0]),
        "seconds": round(seconds, 2),
    }


In [ ]:
import json
import random
import statistics

rows = [json.loads(line) for line in DATA_PATH.open(encoding="utf-8") if line.strip()]
by_paper = {}
for row in rows:
    pid = str(row["paper_id"])
    by_paper.setdefault(pid, {
        "paper_id": pid,
        "whole_paper": row["whole_paper"],
        "reference_review": row.get("review_content", ""),
    })

papers = list(by_paper.values())
random.Random(SEED).shuffle(papers)
if PAPER_IDS:
    wanted = {str(x) for x in PAPER_IDS}
    papers = [p for p in papers if p["paper_id"] in wanted]
    missing = wanted - {p["paper_id"] for p in papers}
    if missing:
        print("paper ids not found in the test split:", sorted(missing))
else:
    papers = papers[:N_PAPERS]
assert papers, "no papers loaded"
if not PAPER_IDS and len(papers) < N_PAPERS:
    print(f"warning: only {len(papers)} papers available; attach the full test split for more")

prompt_lengths = []
for p in papers:
    enc = render(build_prompt(p["whole_paper"]))
    prompt_lengths.append(int(enc["input_ids"].shape[1]))

print(f"loaded {len(rows)} rows -> {len(by_paper)} unique papers; selected {len(papers)}")
print("prompt tokens (min/median/max):", min(prompt_lengths), int(statistics.median(prompt_lengths)), max(prompt_lengths))
print("max prompt + new tokens:", max(prompt_lengths) + MAX_NEW_TOKENS)
assert max(prompt_lengths) + MAX_NEW_TOKENS < 262_144, "exceeds the model's 262k context"

run_config = {
    "base_model": BASE_MODEL,
    "adapter_dir": str(ADAPTER_DIR),
    "data_path": str(DATA_PATH),
    "n_papers": len(papers),
    "paper_ids": [p["paper_id"] for p in papers],
    "paper_ids_requested": PAPER_IDS,
    "max_paper_tokens": MAX_PAPER_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": DO_SAMPLE,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "top_k": TOP_K,
    "repetition_penalty": REPETITION_PENALTY,
    "seed": SEED,
    "smoke_test": SMOKE_TEST,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
}
CONFIG_PATH.write_text(json.dumps(run_config, indent=2), encoding="utf-8")
print("config ->", CONFIG_PATH)


In [ ]:
def completed_pairs():
    done = set()
    if RESUME and REVIEWS_PATH.exists():
        for line in REVIEWS_PATH.open(encoding="utf-8"):
            try:
                rec = json.loads(line)
                done.add((rec["paper_id"], rec["arm"]))
            except Exception:
                pass
    return done

def run_arm(arm, active_model, selected, done):
    todo = [p for p in selected if (p["paper_id"], arm) not in done]
    print(f"\n=== {arm}: {len(todo)} to generate, {len(selected) - len(todo)} already done ===")
    if not todo:
        return
    with REVIEWS_PATH.open("a", encoding="utf-8") as handle:
        for i, p in enumerate(todo, 1):
            out = generate_review(active_model, build_prompt(p["whole_paper"]), seed_for(p["paper_id"]))
            rec = {
                "paper_id": p["paper_id"],
                "arm": arm,
                "review": out["review"],
                "reference_review": p["reference_review"],
                "prompt_tokens": out["prompt_tokens"],
                "output_tokens": out["output_tokens"],
                "seconds": out["seconds"],
                "words": len(out["review"].split()),
                "rouge_l": round(rouge_l(out["review"], p["reference_review"]), 4),
                "repeat5": round(repeat5(out["review"]), 4),
                "hit_cap": out["output_tokens"] >= MAX_NEW_TOKENS,
                "seed": seed_for(p["paper_id"]),
                "max_new_tokens": MAX_NEW_TOKENS,
                "temperature": TEMPERATURE if DO_SAMPLE else 0.0,
                "top_p": TOP_P if DO_SAMPLE else 1.0,
                "top_k": TOP_K if DO_SAMPLE else 0,
                "repetition_penalty": REPETITION_PENALTY,
            }
            handle.write(json.dumps(rec, ensure_ascii=False) + "\n")
            handle.flush()
            print(
                f"[{i}/{len(todo)}] {arm} {p['paper_id']} "
                f"words={rec['words']} tok={rec['output_tokens']} "
                f"rougeL={rec['rouge_l']:.3f} {rec['seconds']:.0f}s"
            )

if not RESUME and REVIEWS_PATH.exists():
    REVIEWS_PATH.unlink()
    print("RESUME=False: cleared", REVIEWS_PATH)


In [ ]:
if RUN_BASELINE:
    run_arm("base", model, papers, completed_pairs())
else:
    print("baseline skipped")


In [ ]:
if RUN_ADAPTER:
    from peft import PeftModel
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    adapter_model = PeftModel.from_pretrained(model, str(ADAPTER_DIR), is_trainable=False)
    adapter_model.eval()
    adapter_model.generation_config.use_cache = True
    allocated, reserved = gpu_mem()
    print(f"adapter loaded; allocated {allocated:.2f} GiB, reserved {reserved:.2f} GiB")
    run_arm("adapter", adapter_model, papers, completed_pairs())
else:
    print("adapter skipped")


In [ ]:
import pandas as pd

if not REVIEWS_PATH.exists():
    raise FileNotFoundError("no reviews yet; run the baseline and/or adapter cells first")

df = pd.read_json(REVIEWS_PATH, lines=True)
df.to_csv(OUTPUT_DIR / "reviews.csv", index=False)

summary = (
    df.groupby("arm")
      .agg(
          n=("paper_id", "nunique"),
          mean_words=("words", "mean"),
          mean_output_tokens=("output_tokens", "mean"),
          mean_rouge_l=("rouge_l", "mean"),
          mean_repeat5=("repeat5", "mean"),
          mean_seconds=("seconds", "mean"),
          cap_reached=("hit_cap", "sum"),
      )
      .round(3)
)
summary.to_csv(OUTPUT_DIR / "summary_by_arm.csv")
print(summary.to_string())

print("\nfiles:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {p.name}  {p.stat().st_size / 1024:.0f} KiB")


In [ ]:
for arm in ["base", "adapter"]:
    part = df[df["arm"] == arm]
    if part.empty:
        continue
    row = part.iloc[0]
    print("=" * 100)
    print(
        f"{arm} | paper {row['paper_id']} | {row['words']} words | "
        f"ROUGE-L {row['rouge_l']:.3f} | repeat5 {row['repeat5']:.3f}"
    )
    print("-" * 100)
    print(row["review"][:2500])


In [ ]:
import shutil

bundle = shutil.make_archive(str(WORK_DIR / "preprint_critic_outputs"), "zip", OUTPUT_DIR)
print("bundle:", bundle, f"{Path(bundle).stat().st_size / 2**20:.1f} MiB")


## Save the run

1. **Save Version → Save & Run All (Commit)**. Kaggle keeps everything under
   `/kaggle/working/preprint_critic_outputs`: `reviews.jsonl`, `reviews.csv`,
   `summary_by_arm.csv`, `run_config.json`, and the zip bundle.
2. To generate more than 25 papers, attach the full `test_dataset_untokenized.jsonl` and raise
   `N_PAPERS` (up to 172 unique papers). `RESUME = True` skips the `(paper_id, arm)` pairs already in
   `reviews.jsonl`, so a 12-hour session can be continued.
3. To compare a different checkpoint, point `ADAPTER_DIR` at another PEFT adapter.
4. `SMOKE_TEST = True` runs one paper for 64 new tokens; use it after changing the environment.

Expected timing on one T4 with fp16: the adapter takes roughly 30–60 seconds per review; the base
model takes roughly 2–4 minutes because it writes ~2,300 tokens. `N_PAPERS = 25` is about 1.5–2
hours for both arms. The KV cache for a 16k prompt plus 4k output is about 3 GiB, so the 16 GiB T4
fits with room to spare.
